# 02 - Exploratory Data Analysis: Drivers & Customers

This notebook explores driver and customer data to identify patterns and potential fraud actors.

## Objectives
- Analyze driver demographics and delivery performance
- Analyze customer demographics and order behavior
- Identify high-risk drivers and customers
- Detect potential collusion patterns between drivers and customers

## Data Source
- PostgreSQL database: `walmart-delivery-fraud-detection`
- Schema: `walmart_fraud`
- Period: January 2023 - December 2023
- Region: Central Florida

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

import sys
sys.path.insert(0, '..')

from src.database.connection import (
    load_orders, load_drivers, load_customers, 
    get_summary_stats, test_connection
)
from src.features.driver_features import (
    create_driver_features, get_suspicious_drivers, get_driver_risk_scores
)
from src.features.customer_features import (
    create_customer_features, get_suspicious_customers, get_customer_risk_scores
)
from src.features.aggregations import create_driver_customer_matrix

# Settings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

# Plotly default template
import plotly.io as pio
pio.templates.default = 'plotly_white'

print('Libraries loaded successfully!')

## 1. Database Connection and Data Loading

In [ ]:
# Test database connection
if test_connection():
    print('Database connection: OK')
else:
    raise Exception('Database connection failed!')

In [ ]:
# Load data from PostgreSQL
orders = load_orders()
drivers = load_drivers()
customers = load_customers()

print('Data loaded from PostgreSQL:')
print(f'  Orders: {len(orders):,} records')
print(f'  Drivers: {len(drivers):,} records')
print(f'  Customers: {len(customers):,} records')

In [ ]:
# Preview data
print('DRIVERS SAMPLE:')
display(drivers.head())

print('\nCUSTOMERS SAMPLE:')
display(customers.head())

---
## 2. Driver Analysis

Analyzing driver demographics and delivery performance to identify potential fraud patterns.

In [ ]:
# Create driver features
driver_features = create_driver_features(drivers, orders)

print(f'Driver features created: {len(driver_features)} drivers')
print(f'\nFeatures: {list(driver_features.columns)}')
driver_features.head()

In [ ]:
# Driver Overview Statistics
print('=' * 60)
print('DRIVER OVERVIEW')
print('=' * 60)
print(f"Total Drivers: {len(driver_features):,}")
print(f"Active Drivers (with orders): {(driver_features['total_orders'] > 0).sum():,}")
print(f"Average Age: {driver_features['age'].mean():.1f} years")
print(f"Average Trips: {driver_features['trips'].mean():.1f}")
print(f"\nDelivery Stats:")
print(f"  Average Orders per Driver: {driver_features['total_orders'].mean():.1f}")
print(f"  Average Missing Rate: {driver_features['missing_rate'].mean():.2f}%")
print(f"  Drivers with Missing Issues: {(driver_features['orders_with_missing'] > 0).sum():,}")

In [ ]:
# Driver Age Distribution
fig = px.histogram(
    driver_features, 
    x='age',
    nbins=25,
    title='Driver Age Distribution',
    labels={'age': 'Age (years)', 'count': 'Number of Drivers'},
    color_discrete_sequence=['#0071CE']
)
fig.add_vline(x=driver_features['age'].mean(), line_dash='dash', line_color='red',
              annotation_text=f"Mean: {driver_features['age'].mean():.1f}")
fig.update_layout(height=400)
fig.show()

In [ ]:
# Driver Experience Distribution
fig = px.histogram(
    driver_features, 
    x='trips',
    nbins=30,
    title='Driver Experience (Total Trips in 2023)',
    labels={'trips': 'Number of Trips', 'count': 'Number of Drivers'},
    color_discrete_sequence=['#17A2B8']
)
fig.add_vline(x=driver_features['trips'].mean(), line_dash='dash', line_color='red',
              annotation_text=f"Mean: {driver_features['trips'].mean():.1f}")
fig.update_layout(height=400)
fig.show()

In [ ]:
# Driver Missing Rate Distribution
active_drivers = driver_features[driver_features['total_orders'] > 0]

fig = px.histogram(
    active_drivers, 
    x='missing_rate',
    nbins=30,
    title='Distribution of Missing Rate by Driver (Active Drivers Only)',
    labels={'missing_rate': 'Missing Rate (%)', 'count': 'Number of Drivers'},
    color_discrete_sequence=['#DC3545']
)
avg_rate = active_drivers['missing_rate'].mean()
fig.add_vline(x=avg_rate, line_dash='dash', line_color='blue',
              annotation_text=f"Mean: {avg_rate:.2f}%")
fig.update_layout(height=400)
fig.show()

print(f"\nDrivers with missing rate > 20%: {(active_drivers['missing_rate'] > 20).sum()}")
print(f"Drivers with missing rate > 30%: {(active_drivers['missing_rate'] > 30).sum()}")
print(f"Drivers with missing rate > 50%: {(active_drivers['missing_rate'] > 50).sum()}")

In [ ]:
# Missing Rate vs Experience
fig = px.scatter(
    active_drivers,
    x='trips',
    y='missing_rate',
    color='age_group',
    size='total_orders',
    hover_data=['driver_name', 'total_items_missing', 'orders_with_missing'],
    title='Missing Rate vs Driver Experience',
    labels={'trips': 'Total Trips', 'missing_rate': 'Missing Rate (%)', 'age_group': 'Age Group'}
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Missing Rate by Age Group
age_analysis = active_drivers.groupby('age_group', observed=True).agg({
    'missing_rate': 'mean',
    'driver_id': 'count',
    'total_orders': 'sum',
    'total_items_missing': 'sum'
}).reset_index()
age_analysis.columns = ['age_group', 'avg_missing_rate', 'driver_count', 'total_orders', 'total_missing']

fig = px.bar(
    age_analysis,
    x='age_group',
    y='avg_missing_rate',
    title='Average Missing Rate by Driver Age Group',
    labels={'avg_missing_rate': 'Average Missing Rate (%)', 'age_group': 'Age Group'},
    color='avg_missing_rate',
    color_continuous_scale='Reds',
    text='avg_missing_rate'
)
fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig.update_layout(height=400)
fig.show()

print('\nAge Group Details:')
display(age_analysis)

In [ ]:
# Missing Rate by Experience Level
exp_analysis = active_drivers.groupby('experience_level', observed=True).agg({
    'missing_rate': 'mean',
    'driver_id': 'count',
    'total_orders': 'sum',
    'total_items_missing': 'sum'
}).reset_index()
exp_analysis.columns = ['experience_level', 'avg_missing_rate', 'driver_count', 'total_orders', 'total_missing']

fig = px.bar(
    exp_analysis,
    x='experience_level',
    y='avg_missing_rate',
    title='Average Missing Rate by Experience Level',
    labels={'avg_missing_rate': 'Average Missing Rate (%)', 'experience_level': 'Experience Level'},
    color='avg_missing_rate',
    color_continuous_scale='Oranges',
    text='avg_missing_rate'
)
fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig.update_layout(height=400)
fig.show()

In [ ]:
# Identify Suspicious Drivers
suspicious_drivers = get_suspicious_drivers(driver_features, missing_rate_threshold=15, min_orders=5)

print('=' * 60)
print('SUSPICIOUS DRIVERS')
print('=' * 60)
print(f"Total suspicious drivers: {len(suspicious_drivers)}")
print(f"Percentage of active drivers: {len(suspicious_drivers)/len(active_drivers)*100:.1f}%")
print()

# Show top 15 suspicious drivers
display(suspicious_drivers[[
    'driver_id', 'driver_name', 'age', 'trips',
    'total_orders', 'total_items_missing', 'missing_rate', 'pct_orders_with_missing'
]].head(15))

In [ ]:
# Calculate Driver Risk Scores
driver_risk = get_driver_risk_scores(driver_features)

# Risk category distribution
risk_dist = driver_risk['risk_category'].value_counts()

fig = px.pie(
    values=risk_dist.values,
    names=risk_dist.index,
    title='Driver Risk Category Distribution',
    color_discrete_sequence=['#28A745', '#FFC107', '#FD7E14', '#DC3545'],
    hole=0.4
)
fig.update_traces(textposition='outside', textinfo='percent+label+value')
fig.update_layout(height=400)
fig.show()

print('Risk Categories:')
for cat in ['Low', 'Medium', 'High', 'Critical']:
    count = (driver_risk['risk_category'] == cat).sum()
    print(f"  {cat}: {count} drivers ({count/len(driver_risk)*100:.1f}%)")

In [ ]:
# Top High/Critical Risk Drivers
high_risk = driver_risk[driver_risk['risk_category'].isin(['High', 'Critical'])].sort_values('risk_score', ascending=False)

print(f"HIGH/CRITICAL RISK DRIVERS: {len(high_risk)}")
print('=' * 60)
display(high_risk[[
    'driver_id', 'driver_name', 'age', 'total_orders',
    'missing_rate', 'risk_score', 'risk_category'
]].head(10))

---
## 3. Customer Analysis

Analyzing customer demographics and order behavior to identify potential fraud patterns.

In [ ]:
# Create customer features
customer_features = create_customer_features(customers, orders)

print(f'Customer features created: {len(customer_features)} customers')
print(f'\nFeatures: {list(customer_features.columns)}')
customer_features.head()

In [ ]:
# Customer Overview Statistics
active_customers = customer_features[customer_features['total_orders'] > 0]

print('=' * 60)
print('CUSTOMER OVERVIEW')
print('=' * 60)
print(f"Total Customers: {len(customer_features):,}")
print(f"Active Customers (with orders): {len(active_customers):,}")
print(f"Average Age: {customer_features['customer_age'].mean():.1f} years")
print(f"\nOrder Stats:")
print(f"  Average Orders per Customer: {active_customers['total_orders'].mean():.1f}")
print(f"  Average Spending: ${active_customers['total_spent'].mean():,.2f}")
print(f"  Average Missing Rate: {active_customers['missing_rate'].mean():.2f}%")
print(f"  Customers with Missing Issues: {(active_customers['orders_with_missing'] > 0).sum():,}")

In [ ]:
# Customer Age Distribution
fig = px.histogram(
    customer_features, 
    x='customer_age',
    nbins=25,
    title='Customer Age Distribution',
    labels={'customer_age': 'Age (years)', 'count': 'Number of Customers'},
    color_discrete_sequence=['#6F42C1']
)
fig.add_vline(x=customer_features['customer_age'].mean(), line_dash='dash', line_color='red',
              annotation_text=f"Mean: {customer_features['customer_age'].mean():.1f}")
fig.update_layout(height=400)
fig.show()

In [ ]:
# Customer Spending Distribution
fig = px.histogram(
    active_customers, 
    x='total_spent',
    nbins=40,
    title='Customer Total Spending Distribution',
    labels={'total_spent': 'Total Spent ($)', 'count': 'Number of Customers'},
    color_discrete_sequence=['#20C997']
)
fig.update_layout(height=400)
fig.show()

In [ ]:
# Customer Missing Rate Distribution
fig = px.histogram(
    active_customers, 
    x='missing_rate',
    nbins=30,
    title='Distribution of Missing Rate by Customer',
    labels={'missing_rate': 'Missing Rate (%)', 'count': 'Number of Customers'},
    color_discrete_sequence=['#DC3545']
)
avg_rate = active_customers['missing_rate'].mean()
fig.add_vline(x=avg_rate, line_dash='dash', line_color='blue',
              annotation_text=f"Mean: {avg_rate:.2f}%")
fig.update_layout(height=400)
fig.show()

print(f"\nCustomers with missing rate > 30%: {(active_customers['missing_rate'] > 30).sum()}")
print(f"Customers with missing rate > 50%: {(active_customers['missing_rate'] > 50).sum()}")
print(f"Customers with 100% missing rate: {(active_customers['missing_rate'] == 100).sum()}")

In [ ]:
# Missing Rate vs Spending
fig = px.scatter(
    active_customers,
    x='total_spent',
    y='missing_rate',
    color='customer_segment',
    size='total_orders',
    hover_data=['customer_name', 'orders_with_missing', 'total_items_missing'],
    title='Missing Rate vs Total Spending',
    labels={'total_spent': 'Total Spent ($)', 'missing_rate': 'Missing Rate (%)', 'customer_segment': 'Segment'}
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Missing Rate by Age Group
cust_age_analysis = active_customers.groupby('age_group', observed=True).agg({
    'missing_rate': 'mean',
    'customer_id': 'count',
    'total_spent': 'sum',
    'total_items_missing': 'sum'
}).reset_index()
cust_age_analysis.columns = ['age_group', 'avg_missing_rate', 'customer_count', 'total_spent', 'total_missing']

fig = px.bar(
    cust_age_analysis,
    x='age_group',
    y='avg_missing_rate',
    title='Average Missing Rate by Customer Age Group',
    labels={'avg_missing_rate': 'Average Missing Rate (%)', 'age_group': 'Age Group'},
    color='avg_missing_rate',
    color_continuous_scale='Reds',
    text='avg_missing_rate'
)
fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig.update_layout(height=400)
fig.show()

In [ ]:
# Missing Rate by Customer Segment
segment_analysis = active_customers.groupby('customer_segment', observed=True).agg({
    'missing_rate': 'mean',
    'customer_id': 'count',
    'total_spent': 'sum',
    'orders_with_missing': 'sum'
}).reset_index()
segment_analysis.columns = ['segment', 'avg_missing_rate', 'customer_count', 'total_revenue', 'total_issues']

fig = px.bar(
    segment_analysis,
    x='segment',
    y='avg_missing_rate',
    title='Average Missing Rate by Customer Segment',
    labels={'avg_missing_rate': 'Average Missing Rate (%)', 'segment': 'Customer Segment'},
    color='avg_missing_rate',
    color_continuous_scale='Purples',
    text='avg_missing_rate'
)
fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig.update_layout(height=400)
fig.show()

print('\nSegment Details:')
display(segment_analysis)

In [ ]:
# Identify Suspicious Customers
suspicious_customers = get_suspicious_customers(customer_features, missing_rate_threshold=25, min_orders=2)

print('=' * 60)
print('SUSPICIOUS CUSTOMERS')
print('=' * 60)
print(f"Total suspicious customers: {len(suspicious_customers)}")
print(f"Percentage of active customers: {len(suspicious_customers)/len(active_customers)*100:.1f}%")
print()

# Show top 15 suspicious customers
display(suspicious_customers[[
    'customer_id', 'customer_name', 'customer_age', 'customer_segment',
    'total_orders', 'total_spent', 'missing_rate', 'orders_with_missing'
]].head(15))

In [ ]:
# Calculate Customer Risk Scores
customer_risk = get_customer_risk_scores(customer_features)

# Risk category distribution
cust_risk_dist = customer_risk['risk_category'].value_counts()

fig = px.pie(
    values=cust_risk_dist.values,
    names=cust_risk_dist.index,
    title='Customer Risk Category Distribution',
    color_discrete_sequence=['#28A745', '#FFC107', '#FD7E14', '#DC3545'],
    hole=0.4
)
fig.update_traces(textposition='outside', textinfo='percent+label+value')
fig.update_layout(height=400)
fig.show()

print('Risk Categories:')
for cat in ['Low', 'Medium', 'High', 'Critical']:
    count = (customer_risk['risk_category'] == cat).sum()
    print(f"  {cat}: {count} customers ({count/len(customer_risk)*100:.1f}%)")

In [ ]:
# Top High/Critical Risk Customers
high_risk_cust = customer_risk[customer_risk['risk_category'].isin(['High', 'Critical'])].sort_values('risk_score', ascending=False)

print(f"HIGH/CRITICAL RISK CUSTOMERS: {len(high_risk_cust)}")
print('=' * 60)
display(high_risk_cust[[
    'customer_id', 'customer_name', 'customer_age', 'customer_segment',
    'total_orders', 'missing_rate', 'risk_score', 'risk_category'
]].head(10))

---
## 4. Driver-Customer Interaction Analysis

Detecting potential collusion patterns between drivers and customers.

In [ ]:
# Create driver-customer interaction matrix
interactions, suspicious_pairs = create_driver_customer_matrix(orders)

print('=' * 60)
print('DRIVER-CUSTOMER INTERACTIONS')
print('=' * 60)
print(f"Total unique driver-customer pairs: {len(interactions):,}")
print(f"Average interactions per pair: {interactions['interactions'].mean():.2f}")
print(f"Max interactions for a pair: {interactions['interactions'].max()}")
print(f"\nSuspicious pairs (high interaction + high missing rate): {len(suspicious_pairs)}")

In [ ]:
# Interaction Distribution
fig = px.histogram(
    interactions,
    x='interactions',
    nbins=20,
    title='Distribution of Driver-Customer Interactions',
    labels={'interactions': 'Number of Interactions', 'count': 'Number of Pairs'},
    color_discrete_sequence=['#6610F2']
)
fig.update_layout(height=400)
fig.show()

# Stats
print(f"Pairs with only 1 interaction: {(interactions['interactions'] == 1).sum()} ({(interactions['interactions'] == 1).sum()/len(interactions)*100:.1f}%)")
print(f"Pairs with 2+ interactions: {(interactions['interactions'] >= 2).sum()}")
print(f"Pairs with 3+ interactions: {(interactions['interactions'] >= 3).sum()}")

In [ ]:
# Repeated Interactions Analysis
repeated = interactions[interactions['interactions'] >= 2].copy()

fig = px.scatter(
    repeated,
    x='interactions',
    y='missing_rate',
    size='items_missing',
    color='missing_rate',
    color_continuous_scale='Reds',
    hover_data=['driver_id', 'customer_id', 'items_missing'],
    title='Missing Rate in Repeated Driver-Customer Interactions',
    labels={'interactions': 'Number of Interactions', 'missing_rate': 'Missing Rate (%)'}
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Suspicious Pairs (Potential Collusion)
print('=' * 70)
print('SUSPICIOUS DRIVER-CUSTOMER PAIRS (POTENTIAL COLLUSION)')
print('=' * 70)
print(f"Criteria: Above average interactions AND missing rate > 1.5x average")
print(f"Found: {len(suspicious_pairs)} suspicious pairs")
print()

if len(suspicious_pairs) > 0:
    # Enrich with names
    susp_enriched = suspicious_pairs.merge(
        drivers[['driver_id', 'driver_name']], on='driver_id', how='left'
    ).merge(
        customers[['customer_id', 'customer_name']], on='customer_id', how='left'
    )
    
    display(susp_enriched[[
        'driver_id', 'driver_name', 'customer_id', 'customer_name',
        'interactions', 'items_missing', 'missing_rate'
    ]].head(15))

In [ ]:
# Heatmap of Top Interacting Pairs
top_drivers = interactions.groupby('driver_id')['interactions'].sum().nlargest(15).index
top_customers = interactions.groupby('customer_id')['interactions'].sum().nlargest(15).index

matrix_data = interactions[
    interactions['driver_id'].isin(top_drivers) & 
    interactions['customer_id'].isin(top_customers)
]

# Create pivot for heatmap
heatmap_pivot = matrix_data.pivot_table(
    values='missing_rate', 
    index='driver_id', 
    columns='customer_id',
    fill_value=0
)

fig = px.imshow(
    heatmap_pivot,
    title='Missing Rate Heatmap: Top Interacting Drivers vs Customers',
    labels={'color': 'Missing Rate (%)'},
    color_continuous_scale='RdYlGn_r',
    aspect='auto'
)
fig.update_layout(height=500)
fig.show()

---
## 5. Key Findings and Summary

In [ ]:
# Summary Statistics
print('=' * 70)
print('KEY FINDINGS - DRIVERS & CUSTOMERS EDA')
print('=' * 70)

# Driver stats
print(f"\n1. DRIVER ANALYSIS")
print(f"   - Total Drivers: {len(drivers):,}")
print(f"   - Active Drivers: {len(active_drivers):,}")
print(f"   - Average Missing Rate: {active_drivers['missing_rate'].mean():.2f}%")
print(f"   - Suspicious Drivers (>15% missing): {len(suspicious_drivers)}")
print(f"   - High/Critical Risk Drivers: {len(high_risk)}")

# Highest risk age group for drivers
highest_risk_age = age_analysis.loc[age_analysis['avg_missing_rate'].idxmax()]
print(f"   - Highest Risk Age Group: {highest_risk_age['age_group']} ({highest_risk_age['avg_missing_rate']:.2f}%)")

# Customer stats
print(f"\n2. CUSTOMER ANALYSIS")
print(f"   - Total Customers: {len(customers):,}")
print(f"   - Active Customers: {len(active_customers):,}")
print(f"   - Average Missing Rate: {active_customers['missing_rate'].mean():.2f}%")
print(f"   - Suspicious Customers (>25% missing): {len(suspicious_customers)}")
print(f"   - High/Critical Risk Customers: {len(high_risk_cust)}")

# Highest risk segment
highest_risk_seg = segment_analysis.loc[segment_analysis['avg_missing_rate'].idxmax()]
print(f"   - Highest Risk Segment: {highest_risk_seg['segment']} ({highest_risk_seg['avg_missing_rate']:.2f}%)")

# Collusion indicators
print(f"\n3. COLLUSION INDICATORS")
print(f"   - Total Driver-Customer Pairs: {len(interactions):,}")
print(f"   - Pairs with Repeated Interactions: {(interactions['interactions'] >= 2).sum()}")
print(f"   - Suspicious Pairs Detected: {len(suspicious_pairs)}")

# Estimated losses
total_missing_items = orders['items_missing'].sum()
avg_item_value = 15  # Estimated average item value
print(f"\n4. ESTIMATED IMPACT")
print(f"   - Total Items Missing: {total_missing_items:,}")
print(f"   - Estimated Loss (at ${avg_item_value}/item): ${total_missing_items * avg_item_value:,.2f}")

---
## Summary

### Key Insights from Drivers & Customers EDA:

#### Drivers:
1. **Suspicious drivers identified** - A subset of drivers shows significantly higher missing rates than average
2. **Experience matters** - Analysis shows correlation between experience level and missing rate
3. **Age group patterns** - Certain age groups show higher fraud indicators
4. **Risk scoring** - Risk categories help prioritize investigation

#### Customers:
1. **Repeat offenders** - Some customers consistently report missing items
2. **Segment analysis** - Missing rates vary by customer value segment
3. **High-value targets** - Some customers may be gaming the system

#### Collusion:
1. **Suspicious pairs detected** - Driver-customer pairs with high interaction and high missing rates
2. **Pattern detection** - Repeated interactions with consistent missing items suggest possible collusion

### Next Steps:
- Deep dive into fraud patterns (Notebook 03)
- Build ML models for fraud detection (Notebook 04)
- Create monitoring dashboard